# Physics-consistent q=20 multi-plane reconstruction

This notebook is the **canonical** measured z-stack phase-retrieval route.

The previous notebook displayed the total wrapped q=20 phase as ‘Recovered phase at z0’ and then propagated the recovered first-plane field using absolute relative-z labels. That was misleading: the q=20 vortex itself produces the spoke-like wrapped phase, and the propagation cell double-counted the z reference.

The current workflow enforces one physical rule:

> **one complex field at one reference plane must propagate to every measured z plane.**

The measured z-stack therefore constrains the longitudinal evolution. The inferred correction remains a 2-D phase at the plane where it is applied; its 3-D consequences come from propagation, not from independently fitting a phase at each z.


In [ ]:
from pathlib import Path
import json, os, sys
from IPython.display import Image as IPyImage, display

HERE = Path.cwd().resolve()
if not (HERE / 'global_multiplane_retrieval.py').is_file():
    candidate = HERE / 'notebooks' / 'experimental' / 'axicon_aberration_correction'
    if candidate.is_dir():
        HERE = candidate
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

DATA_DIR = Path(os.environ.get('BESSEL_ZSCAN_DATA_DIR', HERE / 'z-scan 2 1010')).expanduser().resolve()
OUTPUT_DIR = HERE / 'outputs' / 'physics_consistent_global_multiplane'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data:', DATA_DIR)
print('Output:', OUTPUT_DIR)


## 1. Global z-stack reconstruction

The solver uses a Matsushima-style band-limited angular-spectrum propagator. Interleaved z planes are held out from the retrieval and used only for validation.

The nominal q=20 conical field is propagated to the same reference plane as the reconstructed field. Residual phase is then defined as

$$\Delta\phi(x,y)=\arg[U_{\rm retrieved}(x,y)U_{\rm nominal}^*(x,y)].$$

This removes the complete nominal complex phase, not merely $20\theta$.


In [ ]:
from global_multiplane_retrieval import run_global_multiplane

global_metrics, global_summary = run_global_multiplane(
    DATA_DIR,
    OUTPUT_DIR,
    z_relative_mm=None,
    wavelength_m=1030e-9,
    pixel_pitch_m=5.5e-6,
    q=20,
    output_n=384,
    iterations=160,
    relaxation=0.65,
    beam_radius_m=2.0e-3,
)
print(json.dumps(global_summary, indent=2))
display(global_metrics)


## 2. Phase decomposition

The first panel below is the **total** reconstructed phase and is expected to look highly wrapped because it contains the q=20 topology and nominal propagation phase. It is **not** an aberration map.

The third panel is the quantity relevant to the residual-wavefront hypothesis. Pixels with weak field support are masked rather than interpreted.


In [ ]:
display(IPyImage(filename=str(OUTPUT_DIR / 'phase_decomposition_residual_not_total.png')))


## 3. Longitudinal consistency

Every model column below comes from one reference-plane complex field propagated to all z planes. No per-plane phase refit is used, and the propagation distances are $z_j-z_{\rm ref}$.

The ‘virtual camera-plane conjugate’ is a counterfactual diagnostic only. It is **not** an SLM2 prediction.


In [ ]:
display(IPyImage(filename=str(OUTPUT_DIR / 'global_one_field_signed_xz_yz.png')))


## 4. Held-out planes and falsification

A retrieved phase should not be trusted merely because it can fit the planes used to recover it. The held-out planes test whether one reconstructed wavefield predicts unseen z positions.

A stronger causal check applies the retrieved residual to the nominal reference field and asks whether that phase-only error **recreates** the measured distortion. If this does not improve held-out agreement over the nominal model, the phase map is not accepted as the physical cause of the observed morphology.


In [ ]:
display(IPyImage(filename=str(OUTPUT_DIR / 'heldout_selected_planes.png')))
display(IPyImage(filename=str(OUTPUT_DIR / 'global_multiplane_metrics_vs_z.png')))


## 5. Hardware boundary

This notebook does **not** export an SLM correction.

A camera-plane residual can only become an SLM2 command after the following are measured:

- camera ↔ SLM optical transform and relay magnification;
- parity / rotation;
- illuminated SLM footprint and beam centre;
- 1030-nm phase/gray LUT;
- the physical plane to which the inferred residual should be back-propagated.

After that mapping exists, the conjugate residual can be propagated/mapped to the SLM plane and tested with a **new post-correction z-scan**. Until then `hardware_ready = False`.

The older constrained q=20 annular modal code remains in the repository as a secondary diagnostic/falsification tool, not as the primary 3-D phase retrieval.
